In [1]:
!python -V
%matplotlib inline

Python 3.13.9


In [2]:
import numpy as np
import pandas as pd
import time

# PRNG

In [3]:
class LinearCongruentialGenerator:
    def __init__(self, x0=69069, a=25214903917, b=11, M=2**48):
        """ initialized using the parameters used by Unix drand48()"""
        self.x_now = x0
        self.a = a
        self.b = b
        self.M = M


    def random(self):
        self.x_now = (self.a * self.x_now + self.b) % self.M
        return self.x_now / self.M


class FibonacciGenerator:
    def __init__(self, x0=69069, lag1=607, lag2=273):
        """
        initalized using the parameters used by Go's (Google's C++ replacement) random number generator.eca
        these parameters yield a period of 2^669.
        """
        self.lag1 = lag1
        self.lag2 = lag2

        initializer = LinearCongruentialGenerator(x0=x0)
        self.state = [initializer.random() for _ in range(lag1)]


    def random(self):
        x_now = (self.state[-self.lag1] + self.state[-self.lag2]) % 1

        self.state.append(x_now)
        self.state.pop(0)

        return x_now


random = FibonacciGenerator()

# Problem 0

## Functions

<font color='blue'>
    Let's first try to use the numpy.random.uniform to generate the random numbers. This is easier to code. Once the code is proven to work correctly, we can switch to an PRNG.
</font>

In [4]:
def calc_problem0(Sigma_t, Sigma_a, L, N_samples, epsilon=1e-15):
    """
    Inputs:
        Sigma_t: total cross section
        Sigma_a: absorption cross section
        L: thickness of the shield
        N_samples: total number of particles to be simulated
    
    Outputs:
        N_ref: number of reflected particles
        N_abs: number of absorbed particles
        N_trn: number of transmitted particles
        R_ref: Relative uncertainty of reflection probability
        R_abs: Relative uncertainty of absorption probability
        R_trn: Relative uncertainty of transmission probability
    """
    
    N_ref = 0       # initialize number of reflected particles
    N_abs = 0       # initialize number of absorbed particles
    N_trn = 0       # initialize number of transmitted particles
    n = 0           # initialize counter of simulated particles
    x_init = 0.0    # initial location of the particle
    mu_init = 1.0   # initial direction cosine of the particle
    
    # Next we need a boolean parameter to determine whether or not to start a new particle.
    # This is necessary because after scattering, the particle's life is continued until
    # being transmitted, absorbed or reflected.
    start_new_sample = True
    
    while (n < N_samples):
        
        if (start_new_sample):
            n = n + 1           # Only increment "n" when a new particle is initiated.
            x = x_init
            mu = mu_init
        
        # Sample the path length
        eta = max(epsilon, random.random())
        r = - np.log(eta) / Sigma_t
        
        # The new location of the particle after travelling
        x = x + r*mu
        
        # Check the location of the particle
        if (x > L): # Particle is transmitted
            N_trn = N_trn + 1
            start_new_sample = True
            continue # Move to the beginning of the "while" loop
            
        elif (x < 0): # Particle is reflected
            N_ref = N_ref + 1
            start_new_sample = True
            continue # Move to the beginning of the "while" loop
            
        else: # (x >= 0) and (x <= L), the particle is still inside the shield, determine the interaction type
            eta = max(epsilon, random.random())
            if (eta <= Sigma_a / Sigma_t):
                # The particle is absorbed
                N_abs = N_abs + 1
                start_new_sample = True
            else:                
                # The particle is scattered, determine the new direction
                eta = max(epsilon, random.random())
                mu_0 = 2.0 * eta - 1.0

                eta = max(epsilon, random.random())
                phi_0 = 2.0 * np.pi * eta
                mu = mu*mu_0 + np.sqrt(1 - mu**2) * np.sqrt(1 - mu_0**2) * np.cos(phi_0)
                start_new_sample = False
    
    # After simulating all the particles, calculate the relative uncertainty
    # Note that in Problem 0 we are not using the relative uncertainties as "tolerance" to
    # terminate the simulation. If we do, we need to calculate the relative uncertainties
    # inside the "while" loop.
    R_ref = 1.0  # Give it a large default value to indicate that when N_ref = 0, R_ref is not usable.
    R_abs = 1.0
    R_trn = 1.0
    if (N_ref > 1):    # Do this to avoid dividing by 0
        R_ref = np.sqrt( 1/N_ref - 1/n )        
    if (N_abs > 1):
        R_abs = np.sqrt( 1/N_abs - 1/n )
    if (N_trn > 1):
        R_trn = np.sqrt( 1/N_trn - 1/n )
    
    return N_ref, N_abs, N_trn, R_ref, R_abs, R_trn


def do_problem0(list_N_samples, absorption_ratio):
    list_N_ref   = np.zeros(list_N_samples.size)
    list_N_abs   = np.zeros(list_N_samples.size)
    list_N_trn   = np.zeros(list_N_samples.size)
    list_R_ref   = np.zeros(list_N_samples.size)
    list_R_abs   = np.zeros(list_N_samples.size)
    list_R_trn   = np.zeros(list_N_samples.size)
    list_FOM_ref = np.zeros(list_N_samples.size)
    list_FOM_abs = np.zeros(list_N_samples.size)
    list_FOM_trn = np.zeros(list_N_samples.size)

    Sigma_a = absorption_ratio * Sigma_t
    
    for n in np.arange(list_N_samples.size):
        start = time.time()
        list_N_ref[n], list_N_abs[n], list_N_trn[n], list_R_ref[n], list_R_abs[n], list_R_trn[n] = \
                   calc_problem0(Sigma_t, Sigma_a, L, list_N_samples[n])
        end = time.time()
        T = end - start    # time (in seconds) elapsed for the simulation time
        list_FOM_ref[n] = 1 / (list_R_ref[n] ** 2 * T)
        list_FOM_abs[n] = 1 / (list_R_abs[n] ** 2 * T)
        list_FOM_trn[n] = 1 / (list_R_trn[n] ** 2 * T)

    results_df = pd.DataFrame({
        'N_samples': list_N_samples,
        'P_ref':     list_N_ref/list_N_samples,
        'R_ref':     list_R_ref,
        'FOM_ref':   list_FOM_ref,
        'P_abs':     list_N_abs/list_N_samples,
        'R_abs':     list_R_abs,
        'FOM_abs':   list_FOM_abs,
        'P_trn':     list_N_trn/list_N_samples,
        'R_trn':     list_R_trn,
        'FOM_trn':   list_FOM_trn
    })

    return results_df

## Part A: Purely Absorbing Shield

Assume the current neutron population is related to the initial population like some exponential decay. \
$N = N_0 \exp({-\lambda})$

The input for our relationship is the current distance from the initial location, $x$, which we assume is proportional to $\lambda$ because that is the only free variable. \
$x \propto \lambda$

We expect each material to have some quantity describing neutron interacts inside the material. We also know $\lambda$ is dimensionless, so we need to multiply $x$ by some quantity with a unit of [1 / length]. We will call the quantity describing neutron interaction "cross section," $\Sigma$, and assume it can be expressed with units of [1 / length]. \
$ [x \Sigma] = 1$

From a unit analysis, we can see $\lambda$ has the same units as $x \Sigma$. Also, $N / N_0$ is the fraction of unabsorbed neutrons. So, we arrive at the final expression: \
$ p_{abs} = \exp({-x \Sigma}) = \exp({- 1.0\cdot 5.0}) = 0.00673795$

In [5]:
list_N_samples = np.array([1.0e4, 5.0e4, 1.0e5, 5.0e5, 1.0e6]).astype(int)
Sigma_t = 1
L = 5
absorption_ratio = 1

results_p0cA = do_problem0(list_N_samples, absorption_ratio)
display(results_p0cA)

,N_samples,P_ref,R_ref,FOM_ref,P_abs,R_abs,FOM_abs,P_trn,R_trn,FOM_trn
0,10000,0.0,1.0,62.907640,0.993000,0.000840,8.923898e+07,0.007000,0.119104,4434.576849
1,50000,0.0,1.0,10.456091,0.993740,0.000355,8.299230e+07,0.006260,0.056346,3293.372948
2,100000,0.0,1.0,7.051988,0.993450,0.000257,1.069587e+08,0.006550,0.038945,4649.506572
3,500000,0.0,1.0,0.983636,0.993368,0.000116,7.366650e+07,0.006632,0.017308,3283.512796
4,1000000,0.0,1.0,0.631375,0.993246,0.000082,9.285029e+07,0.006754,0.012127,4293.304699


## Case B: Isotropic Scattering

In [6]:
list_N_samples = np.array([1.0e4, 5.0e4, 1.0e5, 5.0e5, 1.0e6]).astype(int)
Sigma_t = 1
L = 5
absorption_ratio = 0.9

results_p0cA = do_problem0(list_N_samples, absorption_ratio)
display(results_p0cA)

,N_samples,P_ref,R_ref,FOM_ref,P_abs,R_abs,FOM_abs,P_trn,R_trn,FOM_trn
0,10000,0.016000,0.078422,8832.146007,0.976800,0.001541,2.286962e+07,0.007200,0.117426,3939.236757
1,50000,0.016720,0.034295,9262.572728,0.976140,0.000699,2.228508e+07,0.007140,0.052736,3917.263275
2,100000,0.016210,0.024635,8891.185480,0.976550,0.000490,2.247143e+07,0.007240,0.037030,3935.259309
3,500000,0.016424,0.010944,8708.945104,0.976382,0.000220,2.156111e+07,0.007194,0.016614,3779.206128
4,1000000,0.016570,0.007704,8813.168431,0.976057,0.000157,2.132306e+07,0.007373,0.011603,3885.180176


<font color='blue'>
    It can be seen that for both cases (a) and (b), the FOM is relatively constant and the following equation in the lecture notes is also satisfied. <br>
    $\frac{R_2}{R_1} = \sqrt{\frac{N_1}{N_2}}$
</font>

# Problem 1

## Functions

In [7]:
def calc_problem1(Sigma_ts, Sigma_as, Ls, N_samples, epsilon=1e-15):
    """
    Inputs:
        Sigma_ts: list of total cross sections
        Sigma_a: list of absorption cross sections
        Ls: thicknesses of the shields
        N_samples: total number of particles to be simulated
    
    Outputs:
        N_ref: number of reflected particles
        N_abs: number of absorbed particles
        N_trn: number of transmitted particles
        R_ref: Relative uncertainty of reflection probability
        R_abs: Relative uncertainty of absorption probability
        R_trn: Relative uncertainty of transmission probability
    """
    boundaries = np.insert(np.cumsum(Ls), 0, 0.0)

    N_ref = 0
    N_abs = 0
    N_trn = 0
    n = 0
    x_init = 0.0
    mu_init = 1.0

    start_new_sample = True

    while (n < N_samples):        
        if (start_new_sample):
            n = n + 1
            x = x_init
            mu = mu_init
            region_idx = 0
            start_new_sample = False
        
        eta = max(epsilon, random.random())
        sampled_optical_distance = - np.log(eta)

        collision_occurred = False
        while sampled_optical_distance > 0 and not collision_occurred:

            if mu > 0:  distance_to_boundary = (boundaries[region_idx + 1] - x) / mu
            if mu < 0:  distance_to_boundary = (boundaries[region_idx] - x) / mu
            if mu == 0: distance_to_boundary = float("inf")

            optical_distance_to_boundary = distance_to_boundary * Sigma_ts[region_idx]

            # collision in the current region
            if sampled_optical_distance <= optical_distance_to_boundary:
                x = x + (sampled_optical_distance / Sigma_ts[region_idx]) * mu
                tau = 0
                collision_occurred = True
                
            # collision in some next region
            else:
                x = boundaries[region_idx + 1] if mu > 0 else boundaries[region_idx]
                sampled_optical_distance -= optical_distance_to_boundary

                if mu > 0: region_idx += 1
                if mu < 0: region_idx -= 1

                # check where were going
                if region_idx >= len(Sigma_ts):
                    N_trn += 1
                    start_new_sample = True
                    break

                if region_idx < 0:
                    N_ref += 1
                    start_new_sample = True
                    break

        if start_new_sample: continue

        # determine interaction type
        eta = random.random()

        if eta <= (Sigma_as[region_idx] / Sigma_ts[region_idx]):
            N_abs = N_abs + 1
            start_new_sample = True

        if eta > (Sigma_as[region_idx] / Sigma_ts[region_idx]):
            mu = 2 * random.random() - 1
            start_new_sample = False

    # uncertainties
    R_ref = 1.0
    R_abs = 1.0
    R_trn = 1.0
    if (N_ref > 1): R_ref = np.sqrt(1/N_ref - 1/n)        
    if (N_abs > 1): R_abs = np.sqrt(1/N_abs - 1/n)
    if (N_trn > 1): R_trn = np.sqrt(1/N_trn - 1/n)
    
    return N_ref, N_abs, N_trn, R_ref, R_abs, R_trn


def do_problem1(Sigma_ts, Sigma_as, Ls, list_N_samples):
    list_N_ref   = np.zeros(list_N_samples.size)
    list_N_abs   = np.zeros(list_N_samples.size)
    list_N_trn   = np.zeros(list_N_samples.size)
    list_R_ref   = np.zeros(list_N_samples.size)
    list_R_abs   = np.zeros(list_N_samples.size)
    list_R_trn   = np.zeros(list_N_samples.size)
    list_FOM_ref = np.zeros(list_N_samples.size)
    list_FOM_abs = np.zeros(list_N_samples.size)
    list_FOM_trn = np.zeros(list_N_samples.size)

    for n in np.arange(list_N_samples.size):
        start = time.time()
        list_N_ref[n], list_N_abs[n], list_N_trn[n], list_R_ref[n], list_R_abs[n], list_R_trn[n] = \
                   calc_problem1(Sigma_ts, Sigma_as, Ls, list_N_samples[n])
        end = time.time()
        T = end - start    
        
        list_FOM_ref[n] = 1 / (list_R_ref[n] ** 2 * T) if list_R_ref[n] > 0 else 0
        list_FOM_abs[n] = 1 / (list_R_abs[n] ** 2 * T) if list_R_abs[n] > 0 else 0
        list_FOM_trn[n] = 1 / (list_R_trn[n] ** 2 * T) if list_R_trn[n] > 0 else 0

    results_df = pd.DataFrame({
        'N_samples': list_N_samples,
        'P_ref':     list_N_ref/list_N_samples,
        'R_ref':     list_R_ref,
        'FOM_ref':   list_FOM_ref,
        'P_abs':     list_N_abs/list_N_samples,
        'R_abs':     list_R_abs,
        'FOM_abs':   list_FOM_abs,
        'P_trn':     list_N_trn/list_N_samples,
        'R_trn':     list_R_trn,
        'FOM_trn':   list_FOM_trn
    })

    return results_df

## Results

In [8]:
Sigma_ts = [ 0.1,  10, 100]
Sigma_as = [0.01, 0.1,  10]
Ls =       [ 0.1, 0.1, 0.1]

list_N_samples = np.array([1.0e4, 5.0e4, 1.0e5, 5.0e5, 1.0e6]).astype(int)

results_p1 = do_problem1(Sigma_ts, Sigma_as, Ls, list_N_samples)
display(results_p1)

,N_samples,P_ref,R_ref,FOM_ref,P_abs,R_abs,FOM_abs,P_trn,R_trn,FOM_trn
0,10000,0.529600,0.009425,66220.517084,0.466600,0.010692,51452.161881,0.003700,0.164095,218.435639
1,50000,0.531420,0.004199,70667.503716,0.464140,0.004805,53971.347543,0.004420,0.067118,276.637919
2,100000,0.531390,0.002970,69248.381609,0.464540,0.003395,52979.015633,0.004060,0.049528,248.943425
3,500000,0.529660,0.001333,68906.410652,0.466618,0.001512,53530.031513,0.003720,0.023144,228.473531
4,1000000,0.531037,0.000940,69738.924668,0.465373,0.001072,53609.201573,0.003589,0.016662,221.831895


# Problem 2

## Functions

In [9]:
def calc_problem2(Sigma_t, scattering_ratio, L, theta_degrees, target_R=0.10, epsilon=1e-15, max_samples=1e6):
    absorption_ratio = 1 - scattering_ratio

    theta_radians = np.radians(theta_degrees)
    mu_init = np.cos(theta_radians)
    x_init = 0.0
    
    N_ref = 0
    N_abs = 0
    N_trn = 0
    n = 0

    start_new_sample = True
    
    while n < max_samples:
        if start_new_sample:
            n = n + 1
            x = x_init
            mu = mu_init
            start_new_sample = False
            
        # sample path length
        eta = max(epsilon, random.random())
        r = - np.log(eta) / Sigma_t

        x = x + r * mu

        # check boundaries
        if x > L:
            N_trn += 1
            start_new_sample = True
        elif x < 0:
            N_ref += 1
            start_new_sample = True
        else:
            if random.random() <= absorption_ratio:
                N_abs += 1
                start_new_sample = True
            else:
                mu = 2.0 * random.random() - 1.0
                start_new_sample = False

        # require at least one interaction to calc relative uncertainties
        if start_new_sample and N_ref > 1 and N_abs > 1 and N_trn > 1:
            R_ref = np.sqrt(1/N_ref - 1/n)
            R_abs = np.sqrt(1/N_abs - 1/n)
            R_trn = np.sqrt(1/N_trn - 1/n)
            
            if max(R_ref, R_abs, R_trn) <= target_R:
                break

    R_ref = np.sqrt(1/N_ref - 1/n) if N_ref > 1 else 1.0
    R_abs = np.sqrt(1/N_abs - 1/n) if N_abs > 1 else 1.0
    R_trn = np.sqrt(1/N_trn - 1/n) if N_trn > 1 else 1.0
    
    return n, N_ref, N_abs, N_trn, R_ref, R_abs, R_trn


def do_problem2(scattering_ratio, max_samples=1e6):
    angles_degree = [0, 30, 60]
    row_data = []

    for theta in angles_degree:
        n_hist, N_ref, N_abs, N_trn, R_ref, R_abs, R_trn = calc_problem2(
            Sigma_t=2.0, scattering_ratio=scattering_ratio, L=5.0,
            theta_degrees=theta, max_samples=max_samples,
        )

        row_data.append({
            "θ": f"{int(theta)}°",
            "# of histories": n_hist,
            "Transmission (%)": f"{(N_trn / n_hist) * 100:.4f}%",
            "R_trans": f"{R_trn:.5f}",
            "Reflection (%)": f"{(N_ref / n_hist) * 100:.4f}%",
            "R_ref": f"{R_ref:.5f}",
            "Absorption (%)": f"{(N_abs / n_hist) * 100:.4f}%",
            "R_abs": f"{R_abs:.5f}"
        })

    return pd.DataFrame(row_data)

## Case A

In [10]:
scattering_ratio = 0.2

results_p2cA = do_problem2(scattering_ratio)
display(results_p2cA)

,θ,# of histories,Transmission (%),R_trans,Reflection (%),R_ref,Absorption (%),R_abs
0,0°,1000000,0.0061%,0.12803,3.5471%,0.00521,96.4468%,0.00019
1,30°,1000000,0.0014%,0.26726,3.8170%,0.00502,96.1816%,0.00020
2,60°,1000000,0.0002%,0.70711,5.0744%,0.00433,94.9254%,0.00023


## Case B

In [11]:
scattering_ratio = 0.8

results_p2cB = do_problem2(scattering_ratio)
display(results_p2cB)

,θ,# of histories,Transmission (%),R_trans,Reflection (%),R_ref,Absorption (%),R_abs
0,0°,92343,0.1083%,0.09995,28.4602%,0.00522,71.4315%,0.00208
1,30°,133912,0.0747%,0.09996,30.3408%,0.00414,69.5845%,0.00181
2,60°,269612,0.0371%,0.09998,36.7513%,0.00253,63.2116%,0.00147


# Problem 3

## Functions

In [12]:
def calc_problem3(Sigma_t, scattering_ratio, L, theta_degrees, target_R=0.10, max_samples=1e6, w_cutoff=1e-4, roulette_multiplier=5):
    theta_radians = np.radians(theta_degrees)
    mu_init = np.cos(theta_radians)

    # tracking statistics across all runs
    sum_trn = sum_trn2 = 0.0
    sum_ref = sum_ref2 = 0.0
    sum_abs = sum_abs2 = 0.0
    
    n = 0
    while n < int(max_samples):
        n += 1
        x = 0.0
        mu = mu_init
        w = 1.0  # particles initialize with weight of 1
        
        # single history weights
        w_trn_hist = 0.0
        w_ref_hist = 0.0
        w_abs_hist = 0.0

        history_active = True
        while history_active:
            eta = max(1e-15, random.random())
            r = -np.log(eta) / Sigma_t
            x += r * mu

            # check geometric boundaries
            if x > L:
                w_trn_hist = w
                history_active = False
            elif x < 0:
                w_ref_hist = w
                history_active = False
            else:
                # implicit capture
                w_abs_loss = w * (1.0 - scattering_ratio)
                w_abs_hist += w_abs_loss
                w *= scattering_ratio
                
                # Russian roulette
                if w < w_cutoff:
                    if random.random() <= (1.0 / roulette_multiplier):
                        w *= roulette_multiplier
                    else:
                        history_active = False
                        continue

                mu = 2.0 * random.random() - 1.0

        # accumulate history to global totals
        sum_trn += w_trn_hist
        sum_trn2 += w_trn_hist**2
        
        sum_ref += w_ref_hist
        sum_ref2 += w_ref_hist**2
        
        sum_abs += w_abs_hist
        sum_abs2 += w_abs_hist**2

        # check convergence criteria after 1000 generations
        if n < 1000: continue

        if sum_trn > 0 and sum_ref > 0 and sum_abs > 0:
            mean_trn = sum_trn / n
            mean_ref = sum_ref / n
            mean_abs = sum_abs / n
            
            # need to update relative uncertainty formula because the weights vary 
            R_trn = np.sqrt((sum_trn2 - n * mean_trn**2) / (n * (n - 1))) / mean_trn
            R_ref = np.sqrt((sum_ref2 - n * mean_ref**2) / (n * (n - 1))) / mean_ref
            R_abs = np.sqrt((sum_abs2 - n * mean_abs**2) / (n * (n - 1))) / mean_abs
            
            if max(R_trn, R_ref, R_abs) <= target_R:
                break

    # final relative uncertainty calculations
    mean_trn = sum_trn / n
    mean_ref = sum_ref / n
    mean_abs = sum_abs / n
    
    R_trn = np.sqrt((sum_trn2 - n * mean_trn**2) / (n * (n - 1))) / mean_trn
    R_ref = np.sqrt((sum_ref2 - n * mean_ref**2) / (n * (n - 1))) / mean_ref
    R_abs = np.sqrt((sum_abs2 - n * mean_abs**2) / (n * (n - 1))) / mean_abs

    return n, mean_trn, mean_ref, mean_abs, R_trn, R_ref, R_abs


def do_problem3(scattering_ratio, max_samples=1e6, w_cutoff=1e-4, roulette_multiplier=5):
    angles_degree = [0, 30, 60]
    row_data = []

    for theta in angles_degree:
        n_hist, mean_trn, mean_ref, mean_abs, R_trn, R_ref, R_abs = calc_problem3(
            Sigma_t=2.0, scattering_ratio=scattering_ratio, L=5.0,
            theta_degrees=theta, max_samples=max_samples,
            w_cutoff=w_cutoff, roulette_multiplier=roulette_multiplier
        )

        row_data.append({
            "θ": f"{int(theta)}°",
            "# of histories": n_hist,
            "Transmission (%)": f"{mean_trn * 100:.4f}%",
            "R_trans": f"{R_trn:.5f}",
            "Reflection (%)": f"{mean_ref * 100:.4f}%",
            "R_ref": f"{R_ref:.5f}",
            "Absorption (%)": f"{mean_abs * 100:.4f}%",
            "R_abs": f"{R_abs:.5f}"
        })

    return pd.DataFrame(row_data)

## Case A

In [13]:
scattering_ratio = 0.2

results_p3cA = do_problem3(scattering_ratio)

display(results_p3cA)

,θ,# of histories,Transmission (%),R_trans,Reflection (%),R_ref,Absorption (%),R_abs
0,0°,1000000,0.0048%,0.12780,3.5275%,0.00202,96.4677%,0.00007
1,30°,1000000,0.0013%,0.22402,3.8485%,0.00191,96.1502%,0.00008
2,60°,1000000,0.0002%,0.28411,5.0819%,0.00160,94.9180%,0.00009


## Case B

In [14]:
scattering_ratio = 0.8

results_p3cB = do_problem3(scattering_ratio)
display(results_p3cB)

,θ,# of histories,Transmission (%),R_trans,Reflection (%),R_ref,Absorption (%),R_abs
0,0°,22308,0.1321%,0.09988,28.5560%,0.00718,71.3120%,0.00287
1,30°,23864,0.0981%,0.09981,30.0706%,0.00663,69.8314%,0.00285
2,60°,26738,0.0412%,0.09986,37.2492%,0.00521,62.7096%,0.00309


# Problem 4

In [15]:
import numpy as np
import pandas as pd
import random

def get_cell_index(x, cell_width=1.0, num_cells=5):
    if x <= 0:
        return -1
    
    idx = int(x // cell_width)

    if idx >= num_cells:
        return num_cells
    
    return idx

def calc_problem4(Sigma_t, scattering_ratio, L, theta_degrees, target_R=0.10, max_samples=1e6):
    theta_radians = np.radians(theta_degrees)
    mu_init = np.cos(theta_radians)

    # partitioning geometry
    target_cell_width = 2 / Sigma_t  # aiming for cells 2mfp thick
    num_cells = max(1, int(np.round(L / target_cell_width)))
    cell_width = L / num_cells

    # double importance every 2 mfp in we go
    importances = [2**power for power in range(num_cells)]
    
    sum_trn = sum_trn2 = 0.0
    sum_ref = sum_ref2 = 0.0
    sum_abs = sum_abs2 = 0.0
    
    n = 0
    while n < int(max_samples):
        n += 1
        
        w_trn_hist = 0.0
        w_ref_hist = 0.0
        w_abs_hist = 0.0

        # position, mu, weight, region_idx
        queue = [(0.0, mu_init, 1.0, 0)]

        while len(queue) > 0:
            x, mu, w, current_cell = queue.pop()

            branch_active = True
            while branch_active:
                if abs(mu) < 1e-10:
                    distance_to_boundary = float("inf")
                if mu > 0:
                    next_boundary = (current_cell + 1) * cell_width
                    distance_to_boundary = (next_boundary - x) / mu
                if mu < 0:
                    next_boundary = current_cell * cell_width
                    distance_to_boundary = (next_boundary - x) / mu
                
                eta = max(1e-15, random.random())
                sampled_optical_thickness = -np.log(eta) / Sigma_t

                # if the particle hits a boundary before collision
                if sampled_optical_thickness >= distance_to_boundary:
                    x_boundary = next_boundary

                    if mu > 0: new_cell = current_cell + 1
                    if mu < 0: new_cell = current_cell - 1

                    if new_cell >= num_cells:
                        w_trn_hist += w
                        branch_active = False
                        continue
                    elif new_cell < 0:
                        w_ref_hist += w
                        branch_active = False
                        continue

                    previous_importance = importances[current_cell]
                    new_importance = importances[new_cell]
                    importance_ratio = new_importance / previous_importance

                    # moving forward - geometric splitting
                    if importance_ratio > 1.0:
                        w /= importance_ratio
                        # Add twin branch to the queue (inherits the new cell zone)
                        queue.append((x_boundary, mu, w, new_cell))  
                        current_cell = new_cell
                        x = x_boundary
                        continue

                    # Moving backward - geometric Russian roulette
                    elif importance_ratio < 1.0:
                        if random.random() <= importance_ratio:
                            w /= importance_ratio  # Survives, boost weight
                            current_cell = new_cell
                            x = x_boundary
                            continue
                        else:
                            branch_active = False  # Terminated by roulette
                            continue

                # if the particle collides in the current cell
                x = x + sampled_optical_thickness * mu

                if random.random() <= (1.0 - scattering_ratio):
                    w_abs_hist += w
                    branch_active = False
                else:
                    mu = 2.0 * random.random() - 1.0

        # accumulate scores for all branches on this historTREE 
        sum_trn += w_trn_hist
        sum_trn2 += w_trn_hist**2
        sum_ref += w_ref_hist
        sum_ref2 += w_ref_hist**2
        sum_abs += w_abs_hist
        sum_abs2 += w_abs_hist**2

        if n < 1000: continue

        if sum_trn > 0 and sum_ref > 0 and sum_abs > 0:
            mean_trn = sum_trn / n
            mean_ref = sum_ref / n
            mean_abs = sum_abs / n
            
            R_trn = np.sqrt((sum_trn2 - n * mean_trn**2) / (n * (n - 1))) / mean_trn
            R_ref = np.sqrt((sum_ref2 - n * mean_ref**2) / (n * (n - 1))) / mean_ref
            R_abs = np.sqrt((sum_abs2 - n * mean_abs**2) / (n * (n - 1))) / mean_abs
            
            if max(R_trn, R_ref, R_abs) <= target_R:
                break

    mean_trn = sum_trn / n
    mean_ref = sum_ref / n
    mean_abs = sum_abs / n
    R_trn = np.sqrt((sum_trn2 - n * mean_trn**2) / (n * (n - 1))) / mean_trn if mean_trn > 0 else 1.0
    R_ref = np.sqrt((sum_ref2 - n * mean_ref**2) / (n * (n - 1))) / mean_ref if mean_ref > 0 else 1.0
    R_abs = np.sqrt((sum_abs2 - n * mean_abs**2) / (n * (n - 1))) / mean_abs if mean_abs > 0 else 1.0

    return n, mean_trn, mean_ref, mean_abs, R_trn, R_ref, R_abs


def do_problem4(scattering_ratio, max_samples=1e6):
    angles_degree = [0, 30, 60]
    row_data = []

    for theta in angles_degree:
        n_hist, mean_trn, mean_ref, mean_abs, R_trn, R_ref, R_abs = calc_problem4(
            Sigma_t=2.0, scattering_ratio=scattering_ratio, L=5.0,
            theta_degrees=theta, max_samples=max_samples
        )

        row_data.append({
            "θ": f"{int(theta)}°",
            "# of histories": n_hist,
            "Transmission (%)": f"{mean_trn * 100:.4f}%",
            "R_trans": f"{R_trn:.5f}",
            "Reflection (%)": f"{mean_ref * 100:.4f}%",
            "R_ref": f"{R_ref:.5f}",
            "Absorption (%)": f"{mean_abs * 100:.4f}%",
            "R_abs": f"{R_abs:.5f}"
        })

    return pd.DataFrame(row_data)

## Part A

In [16]:
scattering_ratio = 0.2

results_p4cA = do_problem4(scattering_ratio)
display(results_p4cA)

,θ,# of histories,Transmission (%),R_trans,Reflection (%),R_ref,Absorption (%),R_abs
0,0°,129667,0.0057%,0.09952,3.4110%,0.01478,96.5761%,0.00054
1,30°,393745,0.0018%,0.09951,3.8847%,0.00793,96.1212%,0.00033
2,60°,1000000,0.0002%,0.20621,5.0795%,0.00432,94.9186%,0.00023


## Part B

In [17]:
scattering_ratio = 0.8

results_p4cB = do_problem4(scattering_ratio)
display(results_p4cB)

,θ,# of histories,Transmission (%),R_trans,Reflection (%),R_ref,Absorption (%),R_abs
0,0°,7799,0.1122%,0.09988,29.5422%,0.01752,70.4890%,0.00800
1,30°,11717,0.0789%,0.09980,30.9550%,0.01382,68.9106%,0.00664
2,60°,22648,0.0447%,0.09970,37.1865%,0.00865,62.8414%,0.00531
